# 01. Duomenų gavimas ir paruošimas

Projektas: krūties naviko (WDBC) dvejetainė klasifikacija MATLAB R2026a. Teigiama klasė **M → 1**, neigiama **B → 0**. Šaltinis: UCI ID 17, DOI `10.24432/C5DW2B`.

> **Peržiūros notebook.** Čia nėra vykdomo Python kodo. MATLAB fragmentai — citatos iš modulių. Skaičiai — iš `reports/tables/*.csv` ir `reports/freeze.txt`. Grafikai — `../reports/figures/`. Esamų MATLAB/CSV/PDF failų šis aplankas nekeičia.

**Šio failo etapai:** `get_wdbc` → `load_wdbc` → `make_split` → `make_inner_cv` → `fit_scaler` / `apply_scaler`.


## Kas čia vyksta paprastai

Pirmiausia parsisiunčiami originalūs UCI failai ir patikrinama, kad rinkinys tikrai toks, kokio tikimasi: 569 eilutės, 30 požymių, 212 piktybinių ir 357 gerybiniai, be NaN. Identifikatorius (ID) išmetamas **prieš** skaidymą, kad modelis jo nenaudotų kaip požymio.

Tada visi 569 atvejai **vieną kartą** padalinami į mokymą ir testą (~80/20), išlaikant klasių santykį. Testo eilučių statistikos **nedalyvauja** mokyme, hiperparametrų paieškoje, standartizacijoje, Platt kalibracijoje ar slenksčio parinkime.

Požymiai skalėje skiriasi (plotas šimtais–tūkstančiais, tekstūra dešimtimis). Todėl naudojamas z-score, bet **μ ir σ skaičiuojami tik iš mokymo** (vidiniame CV — tik iš to foldo mokymo eilučių). Testas tik perleidžiamas per jau užfiksuotus μ, σ.


## Techninis kontekstas

Konstantos gyvena `config.m`: `cfg.seed = 42`, `cfg.test_size = 0.20`, vidinis CV `k=5`, `repeats=5` (25 skaidiniai), `cfg.cost = [5 1]`, `cfg.se_target = 0.98`. Log-transform išjungtas.

```matlab
% config.m (fragmentas)
cfg.seed = 42;
cfg.test_size = 0.20;
cfg.cv.k = 5;
cfg.cv.repeats = 5;
cfg.se_target = 0.98;
cfg.cost = [5 1];                 % c_FN : c_FP = 5 : 1
cfg.costMatrix = [0 1; 5 0];      % fitcsvm Cost, eilutė = tikroji klasė [B M]
cfg.prep.log_transform = false;
cfg.meta.doi = '10.24432/C5DW2B';
cfg.meta.uci_id = 17;
```


## `data/get_wdbc.m` — parsisiuntimas

**Paprastai:** jei `wdbc.data` jau guli `data/raw/` ir nėra tuščias, pakartotinai nesiunčiama. Kitu atveju `websave` iš UCI. Fiksuojamas SHA-256.

**Techniškai:** tuščias ar nutrūkęs atsisiuntimas — `error()`, ne tylus tęsimas.

```matlab
% data/get_wdbc.m (fragmentas)
raw.data_path = download_one(cfg.urls.wdbc_data, raw.data_path);
raw.names_path = download_one(cfg.urls.wdbc_names, raw.names_path);
raw.sha256_data = file_sha256(raw.data_path);
```


## `data/load_wdbc.m` — skaitymas ir invariantai

**Paprastai:** skaitoma 32 stulpelių CSV (ID, diagnostika, 30 skaičių). ID pašalinamas. M/B verčiama į 1/0. Jei eilučių ne 569, yra NaN, neigiamos reikšmės ar klasių skaičius ne 212/357 — stabdoma. **Nuliai** `concavity` / `concave_points` **leidžiami** (oficialiame WDBC jie yra; tai ne trūkstamos reikšmės).

**Techniškai:** jokios imputacijos. `error()`, ne korekcija.

```matlab
% data/load_wdbc.m (logika)
% 1) 569 eilutės, 32 stulpeliai
% 2) ID pašalinamas PRIEŠ skaidymą
% 3) y: M→1, B→0; 212 M, 357 B
% 4) 0 NaN; X<0 draudžiama; X==0 leistina
```


## `data/make_split.m` — užrakintas 80/20

**Paprastai:** atsitiktinai, bet atkuriamai (`rng(42)`), 20 % eina į testą taip, kad M ir B dalis teste būtų panaši į visą rinkinį.

**Techniškai:** `cvpartition(y,'HoldOut',0.20,'Stratify',true)`. MATLAB natūraliai davė **456 mokymas** (170 M / 286 B) ir **113 testas** (42 M / 71 B), o ne „idealų“ 455/114. Tai priimta `preregistration.md`; indeksai **nekirpti** ranka. Išvestis — tik `idxTrain`, `idxTest` faile `data/processed/split_idx.mat`.

```matlab
% data/make_split.m (fragmentas)
rng(cfg.seed, 'twister');
cvp = cvpartition(y, 'HoldOut', cfg.test_size, 'Stratify', true);
idxTrain = find(training(cvp));
idxTest  = find(test(cvp));
% persidengimas / nepilnas padengimas → error
% testo M/B vs viso rinkinio lūkestis ±1 atvejis
```

| Dalis | n | M (y=1) | B (y=0) | Šaltinis |
|---|---:|---:|---:|---|
| Visas WDBC | 569 | 212 | 357 | `load_wdbc` invariantai |
| Mokymas | 456 | 170 | 286 | `reports/freeze.txt`, `preregistration.md` |
| Testas | 113 | 42 | 71 | tas pats |


## `data/make_inner_cv.m` — 25 vidiniai skaidiniai

**Paprastai:** hiperparametrai ir slenksčiai **neverinami ant testo**. Mokymo 456 eilutės dar 5 kartus dalijamos į 5 poaibius. Kiekviename folde scaler ir modelis mato tik to foldo mokymą.

**Techniškai:** 5×5 = 25 stratifikuoti `KFold`. Pakartojimo sėkla `cfg.seed + 1000*r`; foldo sėkla `cfg.seed + 1000*r + f`. Jei paduodamos visos 569 žymės — `WDBC:InnerCV:FullSet`.

```matlab
% data/make_inner_cv.m (fragmentas)
k = cfg.cv.k;          % 5
R = cfg.cv.repeats;    % 5
cvInner.nPartitions = R * k;   % 25
for r = 1:R
    seed_r = cfg.seed + 1000 * r;
    rng(seed_r, 'twister');
    cvInner.cv{r} = cvpartition(ytrain, 'KFold', k, 'Stratify', true);
    for f = 1:k
        cvInner.seed_fold(r, f) = cfg.seed + 1000 * r + f;
    end
end
```


## `prep/fit_scaler.m` ir `apply_scaler.m` — Kolokviumo (1)

**Paprastai:** šis modulis standartizuoja požymius **tik iš mokymo dalies**, kad testas nebūtų „pamatytas“ iš anksto. Testo eilutė gauna `(x − μ_train) / σ_train`.

**Techniškai:**

$$
z_{ij} = \frac{x_{ij}-\mu_j}{\sigma_j},\quad
\mu_j=\frac{1}{n_{\mathrm{tr}}}\sum_{i\in\mathrm{Tr}} x_{ij},\quad
\sigma_j^2=\frac{1}{n_{\mathrm{tr}}-1}\sum_{i\in\mathrm{Tr}}(x_{ij}-\mu_j)^2.
$$

`std(..., 0, 1)` MATLAB — daliklis $n-1$. Jei paduodama visa 569×30 matrica — `WDBC:FitScaler:FullMatrix`. `sigma=0` — klaida, ne tylus pakeitimas į 1. Imputacija draudžiama.

```matlab
% prep/fit_scaler.m (fragmentas)
if n == 569
    error('WDBC:FitScaler:FullMatrix', ...
        'fit_scaler: gauta visa 569 eiluciu WDBC matrica.');
end
mu = mean(Xfit, 1);
sigma = std(Xfit, 0, 1);   % 1/(n_tr-1), formulė (1)
```

Vidinio CV metu `tune_cv` **kiekviename** iš 25 foldų kviečia `fit_scaler` iš naujo ant `Xtrain(tr,:)`, tada `apply_scaler` validacijai. Galutinis scaler fitinamas ant visų 456 mokymo eilučių **po** HP pasirinkimo — vis dar be testo.


## Nutekėjimo taisyklė (šio etapo santrauka)

| Žingsnis | Matoma mokymo 456 | Matoma testo 113 |
|---|---|---|
| `load_wdbc` invariantai (viso rinkinio dydžiai) | taip (aprašomieji) | taip (aprašomieji; ne fit) |
| `make_split` indeksai | taip | indeksai saugomi, statistikos neskaičiuojamos |
| `fit_scaler` μ, σ | taip | **ne** |
| vidinis CV / HP | taip (OOF) | **ne** (`idxTest` tik assert) |
| `evaluate_test` metrikos | — | **vieną kartą**, po `freeze.txt` |

Šiame etape atskirų PNG grafikų projekte nėra — ROC ir painiavos matricos atsiranda po testo (žr. 03).

Toliau: [02_modeliu_mokymas.ipynb](02_modeliu_mokymas.ipynb).
